In [ ]:
!pip install super-image rasterio matplotlib numpy

In [ ]:
import numpy as np
import rasterio
import torch
import matplotlib.pyplot as plt
from super_image import HanModel

# Define the SR model. HAN is a 3-channel model, so use RGB bands below.
device = "cuda" if torch.cuda.is_available() else "cpu"
srmodel = HanModel.from_pretrained("eugenesiow/han", scale=4)
srmodel.to(device)
srmodel.eval()

# Set this to your Sentinel-2 RGB GeoTIFF (or a stacked multi-band raster).
S2_PATH = "S2A_MSIL2A_20240817T105031_R051_T31UFU_20240817T143149_B02-B03-B04-B08_masked.tif"

# rasterio band indices are 1-based. The source file stores bands in the
# order B02, B03, B04, B08 (Blue, Green, Red, NIR), so pick the natural-color
# RGB bands in R, G, B order for the 3-channel HAN model.
BAND_INDICES = [3, 2, 1]  # R=B04, G=B03, B=B02

# Read the scene as [channels, height, width]. For Sentinel-2 L2A, values are
# commonly scaled by 10000; adjust REFLECTANCE_SCALE if your file is different.
with rasterio.open(S2_PATH) as src:
    lr = src.read(BAND_INDICES).astype(np.float32)
    profile = src.profile.copy()

REFLECTANCE_SCALE = 10000.0
lr = np.clip(lr / REFLECTANCE_SCALE, 0.0, 1.0)
lr_img = torch.from_numpy(lr)

if lr_img.ndim != 3 or lr_img.shape[0] != 3:
    raise ValueError(f"Expected a 3-band raster, got shape {tuple(lr_img.shape)}")

# Run super-resolution. Output shape is [3, height * 4, width * 4].
with torch.no_grad():
    sr_img = srmodel(lr_img.unsqueeze(0).to(device)).squeeze(0).cpu().clamp(0, 1)

# Save the output as a float32 GeoTIFF, preserving the source CRS and transform.
# The transform is scaled because the pixel size is 4x finer after SR.
output_profile = profile.copy()
output_profile.update(
    driver="GTiff",
    count=3,
    dtype="float32",
    height=sr_img.shape[1],
    width=sr_img.shape[2],
    transform=profile["transform"] * rasterio.Affine.scale(1 / 4, 1 / 4),
    nodata=None,
)

with rasterio.open("sentinel2_han_x4.tif", "w", **output_profile) as dst:
    dst.write(sr_img.numpy())

# Display the super-resolved RGB image.
plt.imshow(sr_img.permute(1, 2, 0).numpy())
plt.axis("off")
plt.show()

# Metrics require a matching HR reference raster, registered to this input
# scene. See the next cell for a guarded example that only runs if you have
# defined `metrics` yourself.

In [ ]:
# This cell is a placeholder: `metrics` was never defined anywhere in the
# original notebook (no import, no assignment), so calling it directly would
# raise a NameError. Wire `metrics` up to whatever benchmarking object or
# library you use (e.g. one that computes PSNR/SSIM/spectral metrics against
# a matching high-resolution reference raster), then remove the guard below.
if "metrics" in dir():
    metrics.plot_triplets()
    metrics.plot_summary()
    metrics.plot_tc()
    metrics.plot_ternary()
else:
    print("Skipping metrics plots: define a `metrics` object with a matching HR reference first.")